Код части 2. Что делать и какие решения уже приняты: [02_detection.md](02_detection.md).

Ниже по ячейке на шаг. Пустую ячейку не удалять: если шаг пропущен, в ней пишется причина.


In [1]:

# Шаг 1. Проверка данных
# Убедиться, что configs/plates.yaml указывает на data/processed/detection.
# Посчитать картинки и метки в train и val. Test для подбора не открывать.
# Если пар нет или путь в конфиге другой — остановиться, обучение не начинать.

from __future__ import annotations

import json
import os
import random
import shutil
import time
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import RTDETR, YOLO

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "plates.yaml").exists():
    if (ROOT.parent / "configs" / "plates.yaml").exists():
        os.chdir(ROOT.parent)
        ROOT = Path.cwd().resolve()
    else:
        raise RuntimeError("Не вижу configs/plates.yaml. Ноутбук нужно открывать из корня репозитория.")

DATA_YAML = ROOT / "configs" / "plates.yaml"
DET_ROOT = (ROOT / "data" / "processed" / "detection").resolve()
PROJECT = ROOT / "runs" / "detector"
PROJECT.mkdir(parents=True, exist_ok=True)
METRICS_PATH = PROJECT / "metrics.json"

SEED = 42
EPOCHS = 50
PATIENCE = 10
# True: если прогон уже сохранил last.pt, не доучивать его, а снять метрики и остановиться.
# На более мощном ПК поставь False, тогда та же ячейка продолжит обучение с last.pt.
EVAL_STOPPED = True
BATCH = 8
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
RUNS = ("n640", "n640aug", "n960", "s", "rtdetr")
DEVICE = 0 if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))

# Одинаково во всех строках. Меняется только то, что названо в шаге.
# fliplr и flipud выключены: отражённый номер нельзя отдать в распознавание.
# hsv = 0, чтобы яркость была только в строке с аугментациями, а не в обеих.
COMMON = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    seed=SEED,
    project=str(PROJECT),
    exist_ok=True,
    pretrained=True,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    mosaic=1.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    bgr=0.0,
    plots=True,
    workers=0,
    device=DEVICE,
    cache=False,
    amp=bool(torch.cuda.is_available()),
    verbose=True,
    val=True,
)

DATA_OK = False


def image_stems(split):
    folder = DET_ROOT / split / "images"
    return {p.stem for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT}


def check_split(split):
    images = DET_ROOT / split / "images"
    labels = DET_ROOT / split / "labels"
    if not images.is_dir() or not labels.is_dir():
        raise RuntimeError(f"Нет папки {split}/images или {split}/labels. Обучение не начинаю.")
    imgs = image_stems(split)
    labs = {p.stem for p in labels.iterdir() if p.is_file() and p.suffix == ".txt"}
    missing_labels = sorted(imgs - labs)
    missing_images = sorted(labs - imgs)
    if missing_labels or missing_images:
        raise RuntimeError(
            f"{split}: пары картинка+метка не сходятся. "
            f"Без метки: {len(missing_labels)}, без картинки: {len(missing_images)}. "
            "Обучение не начинаю."
        )
    empty = 0
    boxes = 0
    bad = []
    for stem in sorted(imgs):
        text = (labels / f"{stem}.txt").read_text(encoding="utf-8").strip()
        if not text:
            empty += 1
            continue
        for line_no, line in enumerate(text.splitlines(), start=1):
            parts = line.split()
            if len(parts) != 5:
                bad.append(f"{stem}.txt:{line_no}")
                continue
            cls_id = int(float(parts[0]))
            x, y, w, h = (float(v) for v in parts[1:])
            if cls_id != 0 or w <= 0 or h <= 0:
                bad.append(f"{stem}.txt:{line_no}")
                continue
            boxes += 1
    if bad:
        raise RuntimeError(f"{split}: битые метки, первые {bad[:5]}. Обучение не начинаю.")
    return len(imgs), len(labs), empty, boxes


cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
yaml_path = Path(cfg["path"]).resolve()
if yaml_path != DET_ROOT:
    raise RuntimeError(
        f"plates.yaml path указывает на {yaml_path}, а данные лежат в {DET_ROOT}. Обучение не начинаю."
    )
if cfg.get("train") != "train/images" or cfg.get("val") != "val/images":
    raise RuntimeError(
        f"В plates.yaml train/val должны быть train/images и val/images, сейчас {cfg.get('train')}, {cfg.get('val')}."
    )
if int(cfg.get("nc", 0)) != 1 or cfg.get("names", {}).get(0) != "license_plate":
    raise RuntimeError(f"В plates.yaml ожидался один класс license_plate, сейчас nc={cfg.get('nc')} names={cfg.get('names')}.")

# Test здесь не открывается: ни список файлов, ни метки.
train_n, train_l, train_empty, train_boxes = check_split("train")
val_n, val_l, val_empty, val_boxes = check_split("val")
print(f"Корень: {ROOT}")
print(f"Конфиг: {DATA_YAML}")
print(f"path в конфиге: {yaml_path}")
print(f"train: картинок {train_n}, меток {train_l}, пустых меток {train_empty}, рамок {train_boxes}")
print(f"val: картинок {val_n}, меток {val_l}, пустых меток {val_empty}, рамок {val_boxes}")
print("test не открывал.")
print(f"Устройство: {DEVICE}. Эпох {EPOCHS}, patience {PATIENCE}, seed {SEED}, стартовый batch {BATCH}.")
print("Горизонтальное и вертикальное отражение выключены во всех строках.")


def load_rows():
    if not METRICS_PATH.exists():
        return {}
    return json.loads(METRICS_PATH.read_text(encoding="utf-8"))


def save_rows(rows):
    METRICS_PATH.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")


def skipped(reason):
    return {
        "precision": None,
        "recall": None,
        "map50": None,
        "map5095": None,
        "ms_per_frame": None,
        "weights": None,
        "imgsz": None,
        "augment": None,
        "batch": None,
        "best_epoch": None,
        "plot": None,
        "reason": reason,
    }


def finished(rows, name):
    return (rows.get(name) or {}).get("map5095") is not None


def best_by_map(rows, names):
    ready = [name for name in names if finished(rows, name)]
    if not ready:
        return None
    return max(ready, key=lambda name: (rows[name]["map5095"], -rows[name]["ms_per_frame"]))


def choose_for_part4(rows):
    ready = [name for name in RUNS if finished(rows, name)]
    if not ready:
        raise RuntimeError("Нет ни одной посчитанной строки, вес для части 4 выбрать нельзя.")
    best = max(ready, key=lambda name: rows[name]["map5095"])
    close = [name for name in ready if rows[best]["map5095"] - rows[name]["map5095"] <= 0.01]
    winner = min(close, key=lambda name: rows[name]["ms_per_frame"])
    if winner == best and len(ready) == 1:
        reason = f"{winner} — единственный посчитанный прогон, его вес забирает часть 4."
    elif winner == best:
        reason = (
            f"{winner} лучше остальных на val по mAP@50-95 ({rows[winner]['map5095']:.4f}) "
            f"при {rows[winner]['ms_per_frame']:.0f} мс на кадр."
        )
    else:
        reason = (
            f"{winner} уступает {best} меньше чем на 0.01 mAP@50-95 "
            f"({rows[winner]['map5095']:.4f} против {rows[best]['map5095']:.4f}) "
            f"и быстрее: {rows[winner]['ms_per_frame']:.0f} мс против {rows[best]['ms_per_frame']:.0f} мс."
        )
    return winner, reason


def print_row(row):
    if row.get("map5095") is None:
        print(f"Прогон не запускался. {row.get('reason', '')}")
        return
    print(
        f"Precision {row['precision']:.4f} | Recall {row['recall']:.4f} | "
        f"mAP@50 {row['map50']:.4f} | mAP@50-95 {row['map5095']:.4f} | "
        f"{row['ms_per_frame']:.1f} мс/кадр | лучшая эпоха {row['best_epoch']} | batch {row['batch']}"
    )
    print(f"Веса: {row['weights']}")
    print(f"График: {row['plot']}")


def plate_augmentations():
    # Только яркость, шум и размытие. Поворот задаётся degrees в train().
    # Отражений здесь нет.
    return [
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.0, p=0.5),
        A.GaussNoise(std_range=(0.02, 0.06), mean_range=(0.0, 0.0), per_channel=True, p=0.4),
        A.Blur(blur_limit=3, p=0.3),
    ]


def drop_incomplete_cache(split):
    # Короткий пробный прогон мог записать кэш не на все кадры. На таком кэше обучать нельзя.
    cache_path = DET_ROOT / split / "labels.cache"
    if not cache_path.exists():
        return
    payload = np.load(cache_path, allow_pickle=True).item()
    cached = len(payload.get("labels", []))
    on_disk = len(image_stems(split))
    if cached != on_disk:
        cache_path.unlink()
        print(f"{split}: удалён неполный кэш меток ({cached} из {on_disk}).")


def _memory_error(exc):
    text = str(exc).lower()
    return isinstance(exc, MemoryError) or "memory" in text or "not enough" in text


def best_val_metrics(run_dir):
    frame = pd.read_csv(run_dir / "results.csv")
    frame.columns = [column.strip() for column in frame.columns]
    precision = frame["metrics/precision(B)"]
    recall = frame["metrics/recall(B)"]
    map50 = frame["metrics/mAP50(B)"]
    map5095 = frame["metrics/mAP50-95(B)"]
    # Так ultralytics выбирает best.pt.
    fitness = 0.1 * map50 + 0.9 * map5095
    valid = fitness.notna()
    if not valid.any():
        raise RuntimeError(f"В {run_dir / 'results.csv'} нет валидных метрик val.")
    index = fitness[valid].idxmax()
    return {
        "precision": float(precision.loc[index]),
        "recall": float(recall.loc[index]),
        "map50": float(map50.loc[index]),
        "map5095": float(map5095.loc[index]),
        "best_epoch": int(frame.loc[index, "epoch"]),
        "epochs_done": int(frame["epoch"].max()),
    }


def _already_scored(existing):
    if existing.get("map5095") is None:
        return False
    # Частичная проверка не мешает доучить прогон, когда EVAL_STOPPED выключен.
    if existing.get("partial") and not EVAL_STOPPED:
        return False
    return True


def load_trained(weights):
    try:
        return YOLO(str(weights))
    except Exception:
        return RTDETR(str(weights))


def frame_time_ms(weights, imgsz, n=20):
    model = load_trained(weights)
    paths = sorted(p for p in (DET_ROOT / "val" / "images").iterdir() if p.suffix.lower() in IMG_EXT)
    sample = paths[: n + 1]
    model.predict(str(sample[0]), imgsz=imgsz, device=DEVICE, verbose=False)
    started = time.perf_counter()
    for path in sample[1:]:
        model.predict(str(path), imgsz=imgsz, device=DEVICE, verbose=False)
    return (time.perf_counter() - started) / (len(sample) - 1) * 1000.0


def collect_record(name, imgsz, use_aug, batch):
    run_dir = PROJECT / name
    args_path = run_dir / "args.yaml"
    if args_path.exists():
        saved_args = yaml.safe_load(args_path.read_text(encoding="utf-8"))
        imgsz = int(saved_args.get("imgsz", imgsz))
        batch = int(saved_args.get("batch", batch))
    weights = run_dir / "weights" / "best.pt"
    if not weights.exists():
        weights = run_dir / "weights" / "last.pt"
    metrics = best_val_metrics(run_dir)
    plot = run_dir / "results.png"
    metrics.update(
        {
            "ms_per_frame": frame_time_ms(weights, imgsz),
            "weights": str(weights.resolve()),
            "imgsz": imgsz,
            "augment": bool(use_aug),
            "batch": batch,
            "plot": str(plot.resolve()) if plot.exists() else "",
            "reason": "",
        }
    )
    return metrics


def _pid_alive(pid_text):
    try:
        pid = int(str(pid_text).strip())
    except (TypeError, ValueError):
        return False
    if pid <= 0:
        return False
    import ctypes
    from ctypes import wintypes

    kernel = ctypes.WinDLL("kernel32", use_last_error=True)
    kernel.OpenProcess.argtypes = [wintypes.DWORD, wintypes.BOOL, wintypes.DWORD]
    kernel.OpenProcess.restype = wintypes.HANDLE
    kernel.GetExitCodeProcess.argtypes = [wintypes.HANDLE, ctypes.POINTER(wintypes.DWORD)]
    kernel.GetExitCodeProcess.restype = wintypes.BOOL
    kernel.CloseHandle.argtypes = [wintypes.HANDLE]
    kernel.CloseHandle.restype = wintypes.BOOL
    handle = kernel.OpenProcess(0x1000, False, pid)
    if not handle:
        return False
    code = wintypes.DWORD()
    ok = kernel.GetExitCodeProcess(handle, ctypes.byref(code))
    kernel.CloseHandle(handle)
    # 259 = процесс ещё работает. Любой другой код значит, что его уже завершили.
    return bool(ok) and code.value == 259


def train_detector(name, model_name, imgsz, use_aug, arch="yolo"):
    if not DATA_OK:
        raise RuntimeError("Шаг 1 не пройден, обучение не начинаю.")
    rows = load_rows()
    existing = rows.get(name) or {}
    if _already_scored(existing):
        print(f"{name}: метрики уже есть, обучение не повторяю.")
        print_row(existing)
        if existing.get("reason"):
            print(existing["reason"])
        return existing
    lock_path = PROJECT / f"{name}.lock"
    if lock_path.exists() and _pid_alive(lock_path.read_text(encoding="utf-8").strip()):
        holder = lock_path.read_text(encoding="utf-8").strip()
        raise RuntimeError(f"{name} уже обучается, pid {holder}. Вторую копию не запускаю.")
    if lock_path.exists():
        lock_path.unlink()
    lock_path.write_text(str(os.getpid()), encoding="utf-8")
    try:
        return _train_detector_locked(name, model_name, imgsz, use_aug, arch)
    finally:
        if lock_path.exists() and lock_path.read_text(encoding="utf-8").strip() == str(os.getpid()):
            lock_path.unlink()


def _train_detector_locked(name, model_name, imgsz, use_aug, arch="yolo"):
    if not DATA_OK:
        raise RuntimeError("Шаг 1 не пройден, обучение не начинаю.")
    rows = load_rows()
    existing = rows.get(name) or {}
    if _already_scored(existing):
        print(f"{name}: метрики уже есть, обучение не повторяю.")
        print_row(existing)
        if existing.get("reason"):
            print(existing["reason"])
        return existing

    for split in ("train", "val"):
        drop_incomplete_cache(split)

    run_dir = PROJECT / name
    done_flag = run_dir / "train_done.txt"
    last_pt = run_dir / "weights" / "last.pt"
    batch = BATCH

    if EVAL_STOPPED and not last_pt.exists() and not done_flag.exists():
        record = skipped(
            "Новый прогон не запускался: EVAL_STOPPED = True. "
            "На более мощном ПК поставь EVAL_STOPPED = False."
        )
        rows = load_rows()
        rows[name] = record
        save_rows(rows)
        print_row(record)
        return record

    def fit(batch_size, resume=False):
        if resume:
            model = load_trained(last_pt)
            model.train(resume=True)
            return
        model = RTDETR(model_name) if arch == "rtdetr" else YOLO(model_name)
        model.train(
            **COMMON,
            imgsz=imgsz,
            batch=batch_size,
            name=name,
            degrees=5.0 if use_aug else 0.0,
            augmentations=plate_augmentations() if use_aug else [],
        )

    if done_flag.exists() and last_pt.exists():
        print(f"{name}: обучение уже закончено, заново считаю метрики val.")
    elif last_pt.exists() and EVAL_STOPPED:
        print(
            f"{name}: обучение не продолжаю. "
            "Беру сохранённые веса и считаю метрики val по уже пройденным эпохам."
        )
    elif last_pt.exists():
        print(f"{name}: продолжаю незаконченный прогон с {last_pt.name}.")
        fit(batch, resume=True)
        done_flag.write_text("ok\n", encoding="utf-8")
    else:
        while True:
            try:
                if run_dir.exists():
                    shutil.rmtree(run_dir)
                print(
                    f"{name}: {model_name}, вход {imgsz}, аугментации {'да' if use_aug else 'нет'}, "
                    f"batch {batch}, fliplr=0, flipud=0."
                )
                fit(batch, resume=False)
                break
            except (MemoryError, RuntimeError) as exc:
                if not _memory_error(exc):
                    raise
                if batch == 1:
                    raise MemoryError(f"{name}: не хватило памяти даже при batch 1. Эпохи и размер входа не менял.") from exc
                print(f"{name}: batch {batch} не влез в память, уменьшаю до {batch // 2}. Эпохи и размер картинки те же.")
                batch //= 2
                if run_dir.exists():
                    shutil.rmtree(run_dir, ignore_errors=True)
        done_flag.write_text("ok\n", encoding="utf-8")

    record = collect_record(name, imgsz, use_aug, batch)
    if EVAL_STOPPED and last_pt.exists() and not done_flag.exists():
        epochs_done = int(record.get("epochs_done") or 0)
        record["partial"] = True
        record["reason"] = (
            f"Остановлено после {epochs_done} эпох из {EPOCHS}. "
            f"Лучший чекпоинт — эпоха {record['best_epoch']}. "
            "Полный прогон не продолжался. Чтобы доучить на другом ПК, поставь EVAL_STOPPED = False."
        )
    rows = load_rows()
    rows[name] = record
    save_rows(rows)
    print_row(record)
    if record.get("reason"):
        print(record["reason"])
    return record


def yolo_iou(a, b):
    ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2
    ax2, ay2 = a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2
    bx2, by2 = b[0] + b[2] / 2, b[1] + b[3] / 2
    inter = max(0.0, min(ax2, bx2) - max(ax1, bx1)) * max(0.0, min(ay2, by2) - max(ay1, by1))
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0


def gt_cover(gt, pred):
    ax1, ay1 = gt[0] - gt[2] / 2, gt[1] - gt[3] / 2
    ax2, ay2 = gt[0] + gt[2] / 2, gt[1] + gt[3] / 2
    bx1, by1 = pred[0] - pred[2] / 2, pred[1] - pred[3] / 2
    bx2, by2 = pred[0] + pred[2] / 2, pred[1] + pred[3] / 2
    inter = max(0.0, min(ax2, bx2) - max(ax1, bx1)) * max(0.0, min(ay2, by2) - max(ay1, by1))
    area = gt[2] * gt[3]
    return inter / area if area > 0 else 0.0


def classify_frame(gt, pred, iou_ok=0.5, extra_ratio=1.5):
    used_g, used_p = set(), set()
    candidates = []
    for i, box_g in enumerate(gt):
        for j, box_p in enumerate(pred):
            value = yolo_iou(box_g, box_p)
            if value >= iou_ok:
                candidates.append((value, i, j))
    candidates.sort(reverse=True)
    ok, extra = [], []
    for _, i, j in candidates:
        if i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        gt_area = gt[i][2] * gt[i][3]
        ratio = (pred[j][2] * pred[j][3]) / gt_area if gt_area else 0.0
        if ratio > extra_ratio:
            extra.append((i, j))
        else:
            ok.append((i, j))
    for i, box_g in enumerate(gt):
        if i in used_g:
            continue
        for j, box_p in enumerate(pred):
            if j in used_p:
                continue
            gt_area = box_g[2] * box_g[3]
            ratio = (box_p[2] * box_p[3]) / gt_area if gt_area else 0.0
            if gt_cover(box_g, box_p) >= 0.8 and ratio > extra_ratio:
                used_g.add(i)
                used_p.add(j)
                extra.append((i, j))
                break
    missed = [i for i in range(len(gt)) if i not in used_g]
    false = [j for j in range(len(pred)) if j not in used_p]
    return ok, extra, missed, false


def read_yolo_labels(split):
    if split == "test":
        raise RuntimeError("Test открывается один раз в шаге 8 и только у уже выбранной модели.")
    label_dir = DET_ROOT / split / "labels"
    image_dir = DET_ROOT / split / "images"
    images, boxes = {}, {}
    for path in image_dir.iterdir():
        if path.suffix.lower() not in IMG_EXT:
            continue
        parsed = []
        text = (label_dir / f"{path.stem}.txt").read_text(encoding="utf-8").strip()
        if text:
            for line in text.splitlines():
                parts = line.split()
                parsed.append(tuple(float(v) for v in parts[1:5]))
        images[path.stem] = path
        boxes[path.stem] = parsed
    return images, boxes


def predict_split(weights, imgsz, split="val", conf=0.25):
    if split == "test":
        raise RuntimeError("Предсказания на test из разбора ошибок не делаю.")
    model = load_trained(weights)
    found = {}
    results = model.predict(
        source=str(DET_ROOT / split / "images"),
        imgsz=imgsz,
        conf=conf,
        device=DEVICE,
        batch=8,
        stream=True,
        verbose=False,
    )
    for result in results:
        stem = Path(result.path).stem
        boxes = []
        if result.boxes is not None and len(result.boxes):
            for box in result.boxes.xywhn.cpu().numpy():
                boxes.append(tuple(float(v) for v in box))
        found[stem] = boxes
    return found


def epoch_seconds(name):
    frame = pd.read_csv(PROJECT / name / "results.csv")
    frame.columns = [column.strip() for column in frame.columns]
    times = frame["time"].astype(float).tolist()
    if not times:
        raise RuntimeError(f"В прогоне {name} нет колонки time.")
    durations = [times[0]] + [b - a for a, b in zip(times, times[1:])]
    return float(np.median(durations))


def seconds_per_blank_frame(model_name, imgsz, arch="yolo", repeats=5):
    model = RTDETR(model_name) if arch == "rtdetr" else YOLO(model_name)
    image = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    model.predict(image, imgsz=imgsz, device=DEVICE, verbose=False)
    started = time.perf_counter()
    for _ in range(repeats):
        model.predict(image, imgsz=imgsz, device=DEVICE, verbose=False)
    return (time.perf_counter() - started) / repeats


DATA_OK = True


c:\Projects\ПМОиНС\AI_B2B_SAAS_unicorn_startup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Корень: C:\Projects\ПМОиНС\AI_B2B_SAAS_unicorn_startup
Конфиг: C:\Projects\ПМОиНС\AI_B2B_SAAS_unicorn_startup\configs\plates.yaml
path в конфиге: C:\Projects\ПМОиНС\AI_B2B_SAAS_unicorn_startup\data\processed\detection
train: картинок 19958, меток 19958, пустых меток 0, рамок 20838
val: картинок 2442, меток 2442, пустых меток 0, рамок 2540
test не открывал.
Устройство: cpu. Эпох 50, patience 10, seed 42, стартовый batch 8.
Горизонтальное и вертикальное отражение выключены во всех строках.


In [3]:

# Шаг 2. База: YOLO11n, вход 640, без дополнительных аугментаций
# Имя прогона: n640.
# Обучить, раннюю остановку оставить включённой, горизонтальное отражение выключить.
# На val записать Precision, Recall, mAP@50, mAP@50–95 и время на кадр.

print(
    "База n640. Дополнительных аугментаций нет: яркость, шум, размытие и поворот выключены. "
    "Мозаика, сдвиг и масштаб оставлены по умолчанию и в следующей строке будут теми же. "
    f"fliplr={COMMON['fliplr']}, flipud={COMMON['flipud']}, seed={SEED}, epochs={EPOCHS}, patience={PATIENCE}."
)
train_detector("n640", "yolo11n.pt", 640, use_aug=False)


База n640. Дополнительных аугментаций нет: яркость, шум, размытие и поворот выключены. Мозаика, сдвиг и масштаб оставлены по умолчанию и в следующей строке будут теми же. fliplr=0.0, flipud=0.0, seed=42, epochs=50, patience=10.


RuntimeError: n640 уже обучается, pid 19136. Вторую копию не запускаю.

In [ ]:

# Шаг 3. Те же YOLO11n и 640, но с аугментациями
# Имя прогона: n640aug.
# Аугментации: яркость, шум, размытие, небольшой поворот.
# Горизонтальное и вертикальное отражение не включать.
# Сравнить с n640 и одной фразой записать, помогли ли аугментации.

print(
    "n640aug: к базе добавляются только яркость, шум, размытие и поворот ±5°. "
    "Отражения не включаются. Вход, модель, seed и ранняя остановка те же."
)
train_detector("n640aug", "yolo11n.pt", 640, use_aug=True)

rows = load_rows()
base = rows["n640"]
aug = rows["n640aug"]
delta = aug["map5095"] - base["map5095"]
if delta > 0:
    sentence = (
        f"Аугментации помогли: mAP@50-95 {aug['map5095']:.4f} против {base['map5095']:.4f} у n640 "
        f"(разница {delta:+.4f})."
    )
else:
    sentence = (
        f"Аугментации не помогли: mAP@50-95 {aug['map5095']:.4f} против {base['map5095']:.4f} у n640 "
        f"(разница {delta:+.4f})."
    )
print(sentence)
rows["aug_sentence"] = sentence
save_rows(rows)


In [ ]:

# Шаг 4. Вход 960
# Имя прогона: n960.
# Запускать только если на val предыдущая модель теряет мелкие номера.
# Взять рецепт победителя шагов 2–3 и поменять только размер входа.
# Если прогон не нужен, не удалять ячейку: написать, почему его нет.

rows = load_rows()
for need in ("n640", "n640aug"):
    if not finished(rows, need):
        raise RuntimeError(f"Шаг 4 рано запускать: нет метрик {need}.")

winner23 = best_by_map(rows, ("n640", "n640aug"))
recipe = rows[winner23]
print(
    f"Победитель шагов 2–3: {winner23}. "
    f"Аугментации {'включены' if recipe['augment'] else 'выключены'}, у n960 меняется только вход на 960."
)

images, gt = read_yolo_labels("val")
areas = [box[2] * box[3] for boxes in gt.values() for box in boxes]
threshold = float(np.quantile(areas, 0.25))
print(f"Мелкий номер: площадь рамки не больше нижней четверти val, порог {threshold:.5f} от кадра.")
print("Считаю предсказания на val, conf=0.25. Test не трогаю.")
pred = predict_split(recipe["weights"], recipe["imgsz"], split="val", conf=0.25)

found_all = found_small = small_n = all_n = 0
for stem, boxes in gt.items():
    ok, extra, _, _ = classify_frame(boxes, pred.get(stem, []))
    hit = {i for i, _ in ok + extra}
    for i, box in enumerate(boxes):
        all_n += 1
        is_small = box[2] * box[3] <= threshold
        if is_small:
            small_n += 1
        if i in hit:
            found_all += 1
            if is_small:
                found_small += 1

recall_all = found_all / all_n if all_n else 0.0
recall_small = found_small / small_n if small_n else 0.0
print(
    f"Val recall при conf=0.25: все номера {recall_all:.4f} ({found_all}/{all_n}), "
    f"мелкие {recall_small:.4f} ({found_small}/{small_n})."
)

loses_small = small_n >= 30 and recall_small + 0.05 < recall_all
if loses_small:
    print("Мелкие номера теряются сильнее остальных хотя бы на 0.05 recall. Запускаю n960.")
    train_detector("n960", "yolo11n.pt", 960, use_aug=bool(recipe["augment"]))
else:
    if small_n < 30:
        reason = f"n960 не запускался: мелких номеров на val слишком мало ({small_n}), по ним вход 960 не решаю."
    else:
        reason = (
            f"n960 не запускался: модель {winner23} не теряет мелкие номера отдельно от остальных. "
            f"Recall мелких {recall_small:.4f} при общем {recall_all:.4f}, разрыв меньше 0.05."
        )
    rows = load_rows()
    rows["n960"] = skipped(reason)
    save_rows(rows)
    print(reason)


In [ ]:

# Шаг 5. YOLO11s на победившем рецепте
# Имя прогона: s.
# Меняется только размер модели. Вход и аугментации как у лучшей YOLO11n.
# Если памяти не хватает, уменьшить batch, а не число эпох и не размер картинки.
# Если прогон не запускался, написать причину.

rows = load_rows()
nano_names = [name for name in ("n640", "n640aug", "n960") if finished(rows, name)]
if not nano_names:
    raise RuntimeError("Шаг 5 рано запускать: нет ни одного готового прогона YOLO11n.")
best_nano = best_by_map(rows, nano_names)
recipe = rows[best_nano]
print(
    f"Лучшая YOLO11n: {best_nano}, вход {recipe['imgsz']}, "
    f"аугментации {'да' if recipe['augment'] else 'нет'}. У s меняется только модель на YOLO11s."
)
try:
    train_detector("s", "yolo11s.pt", int(recipe["imgsz"]), use_aug=bool(recipe["augment"]))
except MemoryError as exc:
    rows = load_rows()
    rows["s"] = skipped(str(exc))
    save_rows(rows)
    print(rows["s"]["reason"])


In [ ]:

# Шаг 6. RT-DETR один раз
# Имя прогона: rtdetr.
# Тот же plates.yaml и тот же рецепт, что у лучшей YOLO. Свою сетку для RT-DETR не подбирать.
# Если не влезает по памяти или времени, написать, что прогон не запускался.

rows = load_rows()
yolo_names = [name for name in ("n640", "n640aug", "n960", "s") if finished(rows, name)]
if not yolo_names or not finished(rows, "n640"):
    raise RuntimeError("Шаг 6 рано запускать: сначала нужна хотя бы база n640 и рецепт лучшей YOLO.")

best_yolo = best_by_map(rows, yolo_names)
recipe = rows[best_yolo]
imgsz = int(recipe["imgsz"])
print(
    f"Лучшая YOLO: {best_yolo}. RT-DETR получает тот же вход {imgsz} и те же аугментации "
    f"({'да' if recipe['augment'] else 'нет'}), без отдельного перебора."
)

try:
    yolo_frame = seconds_per_blank_frame("yolo11n.pt", 640, arch="yolo")
    detr_frame = seconds_per_blank_frame("rtdetr-l.pt", imgsz, arch="rtdetr")
except (MemoryError, RuntimeError) as exc:
    if not _memory_error(exc):
        raise
    rows = load_rows()
    rows["rtdetr"] = skipped("RT-DETR не запускался: пробный кадр не влез в память.")
    save_rows(rows)
    print(rows["rtdetr"]["reason"])
else:
    yolo_epoch = epoch_seconds("n640")
    detr_epoch = yolo_epoch * (detr_frame / yolo_frame)
    # Раньше patience эпох остановка не сработает, даже если лучшей была первая.
    lower_hours = detr_epoch * (PATIENCE + 1) / 3600
    print(
        f"Одна эпоха n640 заняла {yolo_epoch / 3600:.2f} ч. "
        f"Кадр YOLO11n {yolo_frame * 1000:.0f} мс, кадр RT-DETR-L {detr_frame * 1000:.0f} мс. "
        f"Нижняя оценка RT-DETR до ранней остановки: {lower_hours:.1f} ч."
    )
    if lower_hours > 24:
        rows = load_rows()
        rows["rtdetr"] = skipped(
            "RT-DETR не запускался: по времени кадра прогон до ранней остановки занимает больше суток "
            f"(оценка {lower_hours:.1f} ч). Отдельный перебор для него не делался."
        )
        save_rows(rows)
        print(rows["rtdetr"]["reason"])
    else:
        try:
            train_detector(
                "rtdetr",
                "rtdetr-l.pt",
                imgsz,
                use_aug=bool(recipe["augment"]),
                arch="rtdetr",
            )
        except MemoryError as exc:
            rows = load_rows()
            rows["rtdetr"] = skipped(f"RT-DETR не запускался: {exc}")
            save_rows(rows)
            print(rows["rtdetr"]["reason"])


In [ ]:

# Шаг 7. Разбор ошибок на val
# На выбранной к этому моменту модели сохранить примеры:
# номер найден верно;
# номер не найден;
# найдено не номер;
# рамка захватила лишнее.
# По val уже можно решать, какая модель лучше. Test здесь не считать.

import cv2

rows = load_rows()
winner, reason = choose_for_part4(rows)
recipe = rows[winner]
print(f"Для разбора беру {winner}: {reason}")
print("Test не считаю.")

images, gt = read_yolo_labels("val")
pred = predict_split(recipe["weights"], int(recipe["imgsz"]), split="val", conf=0.25)
review = PROJECT / "error_review"
if review.exists():
    shutil.rmtree(review)
review.mkdir(parents=True)

counts = {"ok": 0, "miss": 0, "false": 0, "extra": 0}
saved = {"ok": 0, "miss": 0, "false": 0, "extra": 0}
examples = []

for stem in sorted(images):
    gt_boxes = gt.get(stem, [])
    pred_boxes = pred.get(stem, [])
    ok, extra, missed, false = classify_frame(gt_boxes, pred_boxes)
    counts["ok"] += len(ok)
    counts["extra"] += len(extra)
    counts["miss"] += len(missed)
    counts["false"] += len(false)
    kinds = []
    if ok:
        kinds.append("ok")
    if missed:
        kinds.append("miss")
    if false:
        kinds.append("false")
    if extra:
        kinds.append("extra")
    for kind in kinds:
        if saved[kind] >= 3:
            continue
        image = cv2.imread(str(images[stem]))
        if image is None:
            continue
        height, width = image.shape[:2]

        def draw(box, color, canvas=image, w=width, h=height):
            cx, cy, bw, bh = box
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)

        for box in gt_boxes:
            draw(box, (0, 170, 0))
        for box in pred_boxes:
            draw(box, (0, 0, 220))
        out = review / f"{kind}_{saved[kind] + 1:02d}_{stem}.jpg"
        cv2.imwrite(str(out), image)
        saved[kind] += 1
        examples.append(str(out))

print(
    "На val при conf=0.25 и IoU 0.5: "
    f"верно {counts['ok']}, не найден {counts['miss']}, "
    f"не номер {counts['false']}, рамка с лишним {counts['extra']}."
)
print("Зелёная рамка — разметка, красная — предсказание.")
print(f"Примеры: {review}")
for path in examples:
    print(path)

rows = load_rows()
rows["error_review"] = {"model": winner, "counts": counts, "dir": str(review)}
rows["chosen_preview"] = {"name": winner, "reason": reason}
save_rows(rows)


In [ ]:

# Шаг 8. Test один раз
# Посчитать test только у модели, которую забирает часть 4.
# Этим прогоном модель не выбирать и гиперпараметры не менять.

rows = load_rows()
winner, reason = choose_for_part4(rows)
recipe = rows[winner]
print(f"Test считаю один раз у {winner}. Выбор уже сделан по val: {reason}")

model = load_trained(recipe["weights"])
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=int(recipe["imgsz"]),
    batch=max(1, int(recipe["batch"] or BATCH)),
    device=DEVICE,
    workers=0,
    plots=False,
    verbose=False,
)
scores = metrics.results_dict
test_row = {
    "name": winner,
    "precision": float(scores["metrics/precision(B)"]),
    "recall": float(scores["metrics/recall(B)"]),
    "map50": float(scores["metrics/mAP50(B)"]),
    "map5095": float(scores["metrics/mAP50-95(B)"]),
}
print(
    f"Test {winner}: Precision {test_row['precision']:.4f} | Recall {test_row['recall']:.4f} | "
    f"mAP@50 {test_row['map50']:.4f} | mAP@50-95 {test_row['map5095']:.4f}"
)
print("Эти числа в выбор веса не идут.")
rows = load_rows()
rows["test"] = test_row
save_rows(rows)


In [ ]:

# Шаг 9. Таблица и вес для части 4
# Собрать строки n640, n640aug, n960, s, rtdetr.
# Для незапущенных строк метрики оставить пустыми и указать причину.
# Записать путь к файлу весов и одну фразу, почему этот вес, а не соседний.

rows = load_rows()
winner, reason = choose_for_part4(rows)


def cell(value, digits=4):
    if value is None:
        return ""
    return f"{value:.{digits}f}"


table_rows = []
for name in RUNS:
    row = rows.get(name) or skipped("Прогон не запускался: ячейка этого шага ещё не записывала результат.")
    table_rows.append(
        {
            "прогон": name,
            "Precision": cell(row.get("precision")),
            "Recall": cell(row.get("recall")),
            "mAP@50": cell(row.get("map50")),
            "mAP@50-95": cell(row.get("map5095")),
            "мс/кадр": cell(row.get("ms_per_frame"), 1),
            "вход": "" if row.get("imgsz") is None else str(row["imgsz"]),
            "аугментации": "" if row.get("augment") is None else ("да" if row["augment"] else "нет"),
            "batch": "" if row.get("batch") is None else str(row["batch"]),
            "почему пусто": "" if row.get("map5095") is not None and not row.get("partial") else (row.get("reason") or ""),
        }
    )

table = pd.DataFrame(table_rows)
print(table.to_string(index=False))
table_path = PROJECT / "comparison.csv"
table.to_csv(table_path, index=False)
print(f"Таблица: {table_path}")
if rows.get("aug_sentence"):
    print(rows["aug_sentence"])

src = Path(rows[winner]["weights"])
dest = PROJECT / "part4.pt"
shutil.copy2(src, dest)
print(f"Часть 4 забирает {dest}.")
print(f"Это копия {src}.")
print(reason)
if rows.get("test"):
    test_row = rows["test"]
    print(
        f"Test уже посчитан у {test_row['name']} и в этой фразе не участвует: "
        f"P {test_row['precision']:.4f}, R {test_row['recall']:.4f}, "
        f"mAP@50 {test_row['map50']:.4f}, mAP@50-95 {test_row['map5095']:.4f}."
    )

rows["chosen"] = {"name": winner, "weights": str(dest.resolve()), "source": str(src.resolve()), "reason": reason}
save_rows(rows)


In [1]:
# Просмотр качества уже сохранённого n640.
# Обучение не запускает, lock не трогает, шаги 2–9 для этой ячейки не нужны.

from pathlib import Path
import os
import time

import cv2
import pandas as pd
from ultralytics import YOLO

root = Path.cwd().resolve()
if not (root / "configs" / "plates.yaml").exists() and (root.parent / "configs" / "plates.yaml").exists():
    os.chdir(root.parent)
    root = Path.cwd().resolve()

run_dir = root / "runs" / "detector" / "n640"
weights = run_dir / "weights" / "best.pt"
if not weights.exists():
    raise FileNotFoundError(f"Нет файла {weights}")

metrics = pd.read_csv(run_dir / "results.csv")
metrics.columns = [column.strip() for column in metrics.columns]
table = metrics[
    [
        "epoch",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    ]
].copy()
table.columns = ["эпоха", "Precision", "Recall", "mAP@50", "mAP@50-95"]
fitness = 0.1 * table["mAP@50"] + 0.9 * table["mAP@50-95"]
best_row = table.loc[fitness.idxmax()]
print("Val после каждой сохранённой эпохи. Test не открывал.")
print(table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(
    f"Лучший чекпоинт: эпоха {int(best_row['эпоха'])}, "
    f"Precision {best_row['Precision']:.4f}, Recall {best_row['Recall']:.4f}, "
    f"mAP@50 {best_row['mAP@50']:.4f}, mAP@50-95 {best_row['mAP@50-95']:.4f}."
)
print(f"Веса: {weights}")

model = YOLO(str(weights))
images = [
    path
    for path in sorted((root / "data" / "processed" / "detection" / "val" / "images").iterdir())
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
]
model.predict(str(images[0]), imgsz=640, device="cpu", verbose=False)
started = time.perf_counter()
for path in images[1:21]:
    model.predict(str(path), imgsz=640, device="cpu", verbose=False)
ms = (time.perf_counter() - started) / 20 * 1000
print(f"Время одного кадра на CPU: {ms:.1f} мс")

preview = run_dir / "preview"
preview.mkdir(parents=True, exist_ok=True)
shown = []
for result in model.predict(
    [str(path) for path in images[:8]],
    imgsz=640,
    conf=0.25,
    device="cpu",
    verbose=False,
):
    out = preview / f"{Path(result.path).stem}.jpg"
    cv2.imwrite(str(out), result.plot())
    boxes = 0 if result.boxes is None else len(result.boxes)
    print(f"{out.name}: найдено рамок {boxes}")
    shown.append(out)
print(f"Картинки с рамками: {preview}")


Val после каждой сохранённой эпохи. Test не открывал.
 эпоха  Precision  Recall  mAP@50  mAP@50-95
     1     0.9656  0.9669  0.9854     0.7430
     2     0.9719  0.9676  0.9843     0.7683
     3     0.9665  0.9526  0.9808     0.7680
Лучший чекпоинт: эпоха 2, Precision 0.9719, Recall 0.9676, mAP@50 0.9843, mAP@50-95 0.7683.
Веса: C:\Projects\ПМОиНС\AI_B2B_SAAS_unicorn_startup\runs\detector\n640\weights\best.pt
Время одного кадра на CPU: 66.0 мс
10020.jpg: найдено рамок 1
10024.jpg: найдено рамок 1
10038.jpg: найдено рамок 2
1006.jpg: найдено рамок 1
10067.jpg: найдено рамок 1
10069.jpg: найдено рамок 1
10072.jpg: найдено рамок 1
10081.jpg: найдено рамок 1
Картинки с рамками: C:\Projects\ПМОиНС\AI_B2B_SAAS_unicorn_startup\runs\detector\n640\preview
